# 03 — Aggregate: Tract and CSA Metrics

Loads the cleaned interim file, computes per-tract equity metrics, rolls up to CSA,
optionally merges ACS demographic data, and saves the outputs the app reads.

**Prerequisites:**
- `data/interim/requests_{YEAR}_clean.parquet` (from `02_clean.ipynb`)
- `data/raw/baltimore_tracts.geojson` (census tract boundaries)
- `data/raw/baltimore_csas.geojson` (CSA boundaries — from BNIA or Open Baltimore)
- `data/raw/tract_to_csa.csv` — crosswalk with columns `geoid, csa_name, population`

**Outputs (read by the Streamlit app):**
- `data/processed/tract_metrics_{YEAR}.parquet`
- `data/processed/csa_metrics_{YEAR}.parquet`
- `data/processed/tract_boundaries.geojson` (copy of source file, stable)
- `data/processed/csa_boundaries.geojson` (copy of source file, stable)

In [ ]:
import sys, shutil
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
import geopandas as gpd

from balt311.metrics import aggregate_tract, rollup_to_csa

YEAR      = 2024
RAW_DIR   = Path('..') / 'data' / 'raw'
INTERIM   = Path('..') / 'data' / 'interim'
PROC      = Path('..') / 'data' / 'processed'
PROC.mkdir(exist_ok=True)

IN_FILE       = INTERIM   / f'requests_{YEAR}_clean.parquet'
TRACTS_GEO    = RAW_DIR   / 'baltimore_tracts.geojson'
CSA_GEO       = RAW_DIR   / 'baltimore_csas.geojson'
CROSSWALK     = RAW_DIR   / 'tract_to_csa.csv'

OUT_TRACT     = PROC / f'tract_metrics_{YEAR}.parquet'
OUT_CSA       = PROC / f'csa_metrics_{YEAR}.parquet'

In [ ]:
df = pd.read_parquet(IN_FILE)
print(f'Loaded {len(df):,} rows')
print(f'Tracts with data: {df["tract_geoid"].nunique()}')

## 1. Tract-level metrics

In [ ]:
tract_metrics = aggregate_tract(df, geo_col='tract_geoid')
print(f'Tract rows: {len(tract_metrics)}')
print(tract_metrics.describe())

## 2. (Optional) Merge ACS demographics

Download ACS 5-year estimates for Baltimore City (FIPS: state=24, county=510).
Requires a free Census API key: https://api.census.gov/data/key_signup.html

Key variables: `B19013_001E` (median household income), `B01003_001E` (total population).

In [ ]:
# CENSUS_KEY = 'your-key-here'
# ACS_YEAR   = 2023
#
# import requests
# url = (
#     f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
#     f'?get=B19013_001E,B01003_001E,NAME'
#     f'&for=tract:*&in=state:24+county:510'
#     f'&key={CENSUS_KEY}'
# )
# rows = requests.get(url).json()
# acs = pd.DataFrame(rows[1:], columns=rows[0])
# acs['geoid'] = '24510' + acs['tract'].str.zfill(6)   # match TIGER GEOID format
# acs = acs.rename(columns={'B19013_001E': 'median_hh_income', 'B01003_001E': 'population'})
# acs[['geoid','median_hh_income','population']] = acs[['geoid','median_hh_income','population']].apply(pd.to_numeric, errors='coerce')
#
# tract_metrics = tract_metrics.merge(acs[['geoid','median_hh_income','population']], on='geoid', how='left')
# tract_metrics['requests_per_1k'] = (
#     tract_metrics['total_requests'] / (tract_metrics['population'] / 1000)
# )
print('ACS merge skipped (uncomment above cells to enable)')

## 3. CSA roll-up

In [ ]:
if CROSSWALK.exists():
    xwalk = pd.read_csv(CROSSWALK)
    print(f'Crosswalk: {len(xwalk)} tracts mapped to {xwalk["csa_name"].nunique()} CSAs')
    csa_metrics = rollup_to_csa(tract_metrics, xwalk)
    print(f'CSA rows: {len(csa_metrics)}')
    print(csa_metrics.describe())
else:
    print(f'Crosswalk not found at {CROSSWALK} — skipping CSA roll-up.')
    print('See geo_reference.md for crosswalk construction instructions.')
    csa_metrics = None

## 4. Save outputs

In [ ]:
tract_metrics.to_parquet(OUT_TRACT, index=False)
print(f'Saved tract metrics → {OUT_TRACT}')

if csa_metrics is not None:
    csa_metrics.to_parquet(OUT_CSA, index=False)
    print(f'Saved CSA metrics   → {OUT_CSA}')

# Copy boundary files to processed/ so the app only reads from one directory
shutil.copy(TRACTS_GEO, PROC / 'tract_boundaries.geojson')
print(f'Copied tract boundaries → {PROC / "tract_boundaries.geojson"}')

if CSA_GEO.exists():
    shutil.copy(CSA_GEO, PROC / 'csa_boundaries.geojson')
    print(f'Copied CSA boundaries   → {PROC / "csa_boundaries.geojson"}')